# ⚙️ 02. Feature Engineering (Engenharia de Atributos)

**Objetivo:** Transformar dados brutos e temporais em sinais matemáticos que o modelo consiga aprender.

Para atender às hipóteses de negócio, criaremos:
1.  **Calendário & Feriados (Hipótese 1 e 3):** Flags de fim de semana e feriados nacionais.
2.  **Transformação Cíclica:** Ensinar a continuidade do tempo (ex: Dezembro perto de Janeiro).
3.  **Lags (Memória):** O quanto vendeu ontem, semana passada, mês passado.
4.  **Médias Móveis (Tendência/Promoção):** Capturar tendências de alta/baixa recentes (proxy para Hipótese 2).

## 🛠️ Imports e Configurações

### Pacotes 

In [37]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

raiz = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(f'diretorio raiz: {raiz}')

diretorio raiz: c:\Users\Sergio\Desktop\_ESTUDO\_portifolio_full\retail-demand-forecasting


In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.tools import config_default
from src.features import (fengineering_1_ordenar,
                               fengineering_2_datas_feriados,
                               fengineering_3_lag1_lag7,
                               fengineering_4_cyclical,
                               fengineering_5_lags,
                               fengineering_6_rolling,
                               fengineering_7_final,
                               fengineering_8_gerar_arquivo
                               )

### Configurações gerais

In [22]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

DATA_PATH = '../data/raw/train.csv' 
OUTPUT_PATH = '../data/processed/dados_com_features.csv'


In [23]:
# #🚩
# config = load_config()
# ORDENACAO_BASE_BUSCA = config['ordenacao_base_busca']

# print(f'Ordenação das colunas: {ORDENACAO_BASE_BUSCA}')



retorno = config_default()  
ORDENACAO_BASE_BUSCA = retorno[0]
TARGET_PROJETO = retorno[1]


Ordenação das colunas: ['store', 'item', 'date']


##  ⛁ Carregando base

In [24]:
#carregando base
df = pd.read_csv(DATA_PATH)

In [25]:
# >>> regra1: ordenar + criar sales_log

#🚩
# # Ordenação cronológica é vital para séries temporais
# df = df.sort_values(by=ORDENACAO_BASE_BUSCA, ascending=[True, True, True]).reset_index(drop=True)

# # Log Transformation (Suavizar Outliers)
# df['sales_log'] = np.log1p(df['sales'])

# print(f"Shape inicial: {df.shape}")






print("regra1: Ordenar base e criar campo sales_log...")
df = fengineering_1_ordenar(df, ORDENACAO_BASE_BUSCA)


regra1: Ordenar base e criar campo sales_log...
Shape inicial: (867000, 5)


## 📅 Features de Calendário e Feriados
Atende: **Hipótese 1 (Fim de Semana)** e **Hipótese 3 (Feriados)**

In [ ]:
# APLICAÇÃO UNIFICADA 

print("Regra 2: Verificando feriados...")
df_eng = fengineering_2_datas_feriados(df, TARGET_PROJETO)

Regra 2: Verificando feriados...
Gerando features completas...
Verificando feriados:
             date  is_holiday
0      2013-01-01           1
43989  2013-03-29           1
55243  2013-04-21           1
60113  2013-05-01           1
124915 2013-09-07           1


In [ ]:
# >>>calcula a diferença de valores em momentos atipicos

print("Regra 3: Criar campos: lag1 (1 dia) e lag7 (7 dias)...")
df_eng = fengineering_3_lag1_lag7(df_eng)


Regra 3: Criar campos: lag1 (1 dia) e lag7 (7 dias)...


In [28]:
df_eng.sample(5)

,date,store,item,sales,sales_log,year,month,day,day_of_week,day_name,...,lag_28,lag_91,rolling_mean_7,rolling_std_7,rolling_mean_28,rolling_std_28,rolling_mean_91,rolling_std_91,sales_diff_lag1,sales_diff_lag7
385117,2015-02-10,10,3,36,3.610918,2015,2,10,1,Tuesday,...,3.332205,3.367296,3.399879,0.344155,3.343839,0.239393,3.423686,0.279514,-10.0,6.0
688892,2016-10-09,4,35,86,4.465908,2016,10,9,6,Sunday,...,4.691348,4.795791,4.261829,0.127947,4.342638,0.217778,4.448423,0.200318,11.0,1.0
608331,2016-05-01,5,2,71,4.276666,2016,5,1,6,Sunday,...,4.110874,3.610918,3.899927,0.219793,3.903394,0.194464,3.780322,0.226378,7.0,10.0
776203,2017-04-02,9,50,105,4.663439,2017,4,2,6,Sunday,...,4.330733,4.219508,4.195869,0.201759,4.240509,0.179974,4.102333,0.215509,16.0,12.0
454142,2015-06-28,1,11,78,4.369448,2015,6,28,6,Sunday,...,4.189655,4.248495,4.321994,0.185175,4.324418,0.154971,4.254508,0.180314,-2.0,-2.0


## 🔄 Transformação Cíclica (Seno/Cosseno)
Ensina ao modelo que o final do ano (dia 365) está perto do começo (dia 1).

In [ ]:

print("Regra 4: Criando campos ciclico para data - seno/coseno...")
df_eng = fengineering_4_cyclical(df_eng)


Regra 4: Criando campos ciclico para data - seno/coseno...
Features cíclicas criadas.


## 🕰️ Lags (Memória Temporal)
Essencial para capturar a autocorrelação.

In [ ]:

print("Regra 5: Criando os campos lags entre  1 a 91 ...")
df_eng = fengineering_5_lags(df_eng, TARGET_PROJETO)

Regra 5: Criando os campos lags entre  1 a 91 ...
Gerando Lags


## 📈 Rolling Windows (Tendência e Volatilidade)
Captura o comportamento médio recente (proxy para promoções ou mudanças de mercado).

In [ ]:
print("Regra 6: Criando campo rolling windows entre 7 a 91 ...")
df_eng = fengineering_6_rolling(df_eng, TARGET_PROJETO)

Regra 6: Criando campo rolling windows entre 7 a 91 ...
Gerando Rolling Windows...


## 🧹 Limpeza e Salvamento
Removemos os meses iniciais que ficaram sem histórico (NaN) devido aos lags de 90 dias.

In [32]:
df_eng.head()

,date,store,item,sales,sales_log,year,month,day,day_of_week,day_name,...,lag_28,lag_91,rolling_mean_7,rolling_std_7,rolling_mean_28,rolling_std_28,rolling_mean_91,rolling_std_91,sales_diff_lag1,sales_diff_lag7
45863,2013-04-02,1,1,19,2.995732,2013,4,2,1,Tuesday,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-9.0,3.0
46241,2013-04-03,1,1,24,3.218876,2013,4,3,2,Wednesday,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,-5.0
46708,2013-04-04,1,1,18,2.944439,2013,4,4,3,Thursday,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,2.0
47337,2013-04-05,1,1,19,2.995732,2013,4,5,4,Friday,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-6.0,4.0
47876,2013-04-06,1,1,23,3.178054,2013,4,6,5,Saturday,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0


In [33]:
df_eng.columns

Index(['date', 'store', 'item', 'sales', 'sales_log', 'year', 'month', 'day',
       'day_of_week', 'day_name', 'day_of_year', 'is_weekend', 'is_payday',
       'holiday_name', 'is_holiday', 'days_until_holiday', 'month_sin',
       'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin',
       'day_of_year_cos', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14',
       'lag_21', 'lag_28', 'lag_91', 'rolling_mean_7', 'rolling_std_7',
       'rolling_mean_28', 'rolling_std_28', 'rolling_mean_91',
       'rolling_std_91', 'sales_diff_lag1', 'sales_diff_lag7'],
      dtype='object')

In [ ]:

print("Regra 7: Limpando os campos nulos...")
df_final = fengineering_7_final(df_eng, df)

Regra 7: Limpando os campos nulos...
Shape Original: (867000, 5)
Shape Final (Feature Eng): (776000, 38)


,date,sales,lag_1,lag_91,is_holiday,month_sin
91152,2013-07-02,17,3.178054,2.995732,0,-0.5
91970,2013-07-03,12,2.890372,3.218876,0,-0.5
92195,2013-07-04,24,2.564949,2.944439,0,-0.5
92641,2013-07-05,17,3.218876,2.995732,0,-0.5
93189,2013-07-06,16,2.890372,3.178054,0,-0.5


In [35]:
df_final.columns

Index(['date', 'store', 'item', 'sales', 'sales_log', 'year', 'month', 'day',
       'day_of_week', 'day_name', 'day_of_year', 'is_weekend', 'is_payday',
       'holiday_name', 'is_holiday', 'days_until_holiday', 'month_sin',
       'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin',
       'day_of_year_cos', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14',
       'lag_21', 'lag_28', 'lag_91', 'rolling_mean_7', 'rolling_std_7',
       'rolling_mean_28', 'rolling_std_28', 'rolling_mean_91',
       'rolling_std_91', 'sales_diff_lag1', 'sales_diff_lag7'],
      dtype='object')

## 💾 Salvando o Arquivo

In [ ]:
# # Salvar 

print("Regra 8: Gerando o arquivo final...")
fengineering_8_gerar_arquivo(df_final, OUTPUT_PATH)

Regra 8: Gerando o arquivo final...
Dataset enriquecido salvo em: ../data/processed/dados_com_features.csv
